**Module 1: Environment & Configuration**

In [3]:
# missing library
!pip install lime shap imbalanced-learn

In [4]:

#=============================================================================
# FAKE NEWS DETECTION - CRITICALLY FIXED & COMPLETE ASSIGNMENT CODE
# Datasets: ISOT Fake News + WELFake (Combined for Stronger Training)
# Models: Logistic Regression | LSTM | BERT
# =============================================================================

import os
import re
import random
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from wordcloud import WordCloud
# Auto-detect environment (Jupyter/Colab/Terminal) for smooth progress bars
from tqdm.auto import tqdm

# Machine Learning & Metrics
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef,
                             cohen_kappa_score, classification_report,
                             confusion_matrix, roc_curve, auc)
from imblearn.over_sampling import SMOTE



In [5]:
# Deep Learning & Transformers
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification

# FIX: transformers এর বদলে torch.optim থেকে AdamW নেওয়া নিরাপদ
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup

# Explainability (XAI)
import lime
from lime.lime_text import LimeTextExplainer
import shap

warnings.filterwarnings('ignore')

**SECTION 1. REPRODUCIBILITY SEED & OUTPUT STRUCTURE SETUP**

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. REPRODUCIBILITY SEED & OUTPUT STRUCTURE SETUP
# ─────────────────────────────────────────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

OUTPUT_DIR = "assignment_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

global_roc_data = {}
global_metrics_summary = []
print("[🚀 SUCCESS] Step 1: Environment, Seeds, and Export Directory initialized successfully.")

** SECTION 2. ENVIRONMENT-AWARE GOOGLE DRIVE MOUNT & CLEANING**

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. ENVIRONMENT-AWARE GOOGLE DRIVE MOUNT & CLEANING
# ─────────────────────────────────────────────────────────────────────────────
try:
    print("[MANDATORY] Attempting to mount Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("[INFO] Google Colab environment not detected. Using local directory instead.")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

**Module 2: Data Selection, Splitting, & Processing**

**SECTION 3. DATA LOADING & STRATIFIED SPLITTING STRATEGY**

**1: Directory Setup & Configuration**

In [8]:
#--- DIRECTORY CONFIGURATION ---
# Dataset Drive Link: https://drive.google.com/drive/folders/1kKKOD1x0xJBcEWY9emdrtrFLSYQipkku?usp=drive_link
ISOT_DIR = "/content/drive/MyDrive/Assignment/ISOTFakeNews"
WEL_DIR = "/content/drive/MyDrive/Assignment/WelFakeNews"

if not os.path.exists(ISOT_DIR): ISOT_DIR = "ISOTFakeNews"
if not os.path.exists(WEL_DIR): WEL_DIR = "WelFakeNews"

**2: Data Loading & Preprocessing**

In [9]:
# --- DATA LOADING & CLEANING ---
print("\n[LOADING] Processing datasets...")
# ISOT Processing
true_df = pd.read_csv(os.path.join(ISOT_DIR, 'True.csv'))
fake_df = pd.read_csv(os.path.join(ISOT_DIR, 'Fake.csv'))
true_df['label'], fake_df['label'] = 1, 0
isot_raw = pd.concat([true_df, fake_df], ignore_index=True)
isot_raw['content'] = isot_raw['title'].fillna('') + ' ' + isot_raw['text'].fillna('')
isot_raw = isot_raw[['content', 'label']].dropna()
isot_raw['dataset_origin'] = 'ISOT'

# WELFake Processing
welfake_raw = pd.read_csv(os.path.join(WEL_DIR, 'WELFake_Dataset.csv'))
welfake_raw['content'] = welfake_raw['title'].fillna('') + ' ' + welfake_raw['text'].fillna('')
welfake_raw = welfake_raw[['content', 'label']].dropna()
welfake_raw['label'] = welfake_raw['label'].map({0: 1, 1: 0})
welfake_raw['dataset_origin'] = 'WELFake'

**3: Stratified Splitting Strategy**

In [10]:
#--- TRAIN-VAL SPLITTING ---
# Cross-Domain Stress Testing Setup
welfake_train_split, welfake_generalization_test = train_test_split(
    welfake_raw, test_size=5000, stratify=welfake_raw['label'], random_state=42
)

master_df = pd.concat([isot_raw, welfake_train_split], ignore_index=True)
train_df, val_df = train_test_split(master_df, test_size=0.2, stratify=master_df['label'], random_state=42)

print(f"Master Set Size: {len(master_df):,}")
print(f"Train Size: {len(train_df):,} | Validation Size: {len(val_df):,}")

4: VOCABULARY & DATASET PREP

4a. VOCABULARY PREP

In [11]:
# --- VOCABULARY PREP ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from collections import Counter
counter = Counter()
for text in train_df['content'].values:
    counter.update(clean_text(text).split())

vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in counter.most_common(19998):
    vocab[word] = len(vocab)

4b. DATASET & DATALOADER

In [12]:
# --- DATASET & DATALOADER ---
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=200):
        self.labels, self.max_len, self.vocab = labels, max_len, vocab
        self.data = [self._encode(t) for t in texts]

    def _encode(self, text):
        tokens = clean_text(text).split()[:self.max_len]
        ids = [self.vocab.get(w, 1) for w in tokens]
        ids += [0] * (self.max_len - len(ids))
        return torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.data[i], torch.tensor(self.labels[i], dtype=torch.long)

train_dl = DataLoader(TextDataset(train_df['content'].values, train_df['label'].values, vocab), batch_size=64, shuffle=True)
val_dl = DataLoader(TextDataset(val_df['content'].values, val_df['label'].values, vocab), batch_size=64)

 EDA (Exploratory Data Analysis) and visualization.


1. DATA COUNTS ---

In [13]:
# --- 1. DATA COUNTS ---
isot_count = len(isot_raw)
welfake_count = len(welfake_raw)
holdout_count = len(welfake_generalization_test)
master_count = len(master_df)
train_count = len(train_df)
val_count = len(val_df)

2. GRAPH CONFIGURATION & NODES


In [14]:
# --- 2. GRAPH CONFIGURATION & NODES ---
from graphviz import Digraph

# Set title and graph properties
dot = Digraph("Dataset_Preparation", format="png",
              graph_attr={
                  'label': 'Dataset Preparation',
                  'labelloc': 't',
                  'fontsize': '20',
                  'fontname': 'Arial'
              })

# Define graph layout and style
dot.attr(rankdir="TB", splines="polyline", nodesep="0.7", ranksep="1.0", dpi="300")
dot.attr("node", shape="box", style="rounded,filled", fontname="Times New Roman", fontsize="14", penwidth="1.5")

# Dataset nodes
dot.node("ISOT", f"ISOT Dataset\n{isot_count:,} Samples", fillcolor="lightblue")
dot.node("WEL", f"WELFake Dataset\n{welfake_count:,} Samples", fillcolor="lightgreen")
dot.node("HOLD", f"Cross-Domain Test Set\n{holdout_count:,} Samples", fillcolor="mistyrose")
dot.node("MASTER", f"Master Dataset\n{master_count:,} Samples", fillcolor="khaki")
dot.node("TRAIN", f"Training Set\n{train_count:,} Samples", fillcolor="lavender")
dot.node("VAL", f"Validation Set\n{val_count:,} Samples", fillcolor="lavender")

# Connections
dot.edge("ISOT", "MASTER", label="Merged", fontsize="12")
dot.edge("WEL", "MASTER", label="Training Portion", fontsize="12")
dot.edge("WEL", "HOLD", label=f"{holdout_count:,} Samples", fontsize="12")
dot.edge("MASTER", "TRAIN", label="80%", fontsize="12")
dot.edge("MASTER", "VAL", label="20%", fontsize="12")